In [1]:
!pip install -q joblib

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [3]:
#load disease_data csv file

from google.colab import drive
drive.mount('/content/gdrive')
data = pd.read_csv('/content/gdrive/MyDrive/Omdena/NLP in drug prediction/test/processed_diseases-priority.csv')

# Select only 100 unique diseases
#selected_diseases = data['Disease'].unique()[:100]  # Pick the first 100 unique diseases
#data = data[data['Disease'].isin(selected_diseases)]
#data.to_csv('/content/symptoms_data.csv', index = False)

# Set target sample size per disease
target_sample_size = 50

# Perform random oversampling to ensure each disease
df = data.groupby('Disease', group_keys=False).apply(lambda x: x.sample(target_sample_size, replace=True))

# Reset index
df= df.reset_index(drop=True)

Mounted at /content/gdrive


<ipython-input-3-a0282fa569f3>:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = data.groupby('Disease', group_keys=False).apply(lambda x: x.sample(target_sample_size, replace=True))


In [4]:
# Convert symptoms to a structured format using tfidf
vectorizer_tfidf = TfidfVectorizer()
X = vectorizer_tfidf.fit_transform(df["Symptoms"])  # Convert symptoms into a numerical format

# Encode the target variable (disease)
y = df["Disease"].str.lower()

# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predict on the test set
y_pred = clf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(accuracy)


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0.8908673894912427


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [5]:
###Export RF model and vectorizer_tfidf
joblib.dump(clf, "/content/disease_model.pkl")
joblib.dump(vectorizer_tfidf, "/content/tfidf_vectorizer.pkl")

['/content/tfidf_vectorizer.pkl']

In [6]:
!pip install -q streamlit
!pip install -q joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.2 MB/s eta 0:00:00


In [5]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import os
import requests

# 🔹 Set Google Drive path (Modify if necessary)
drive_path = "/content/"


# 🔹 Load the trained model and vectorizer
@st.cache_resource
def load_model():
    model_path = os.path.join(drive_path, "disease_model.pkl")
    vectorizer_path = os.path.join(drive_path, "tfidf_vectorizer.pkl")

    if not os.path.exists(model_path) or not os.path.exists(vectorizer_path):
        st.error("🚨 Model or vectorizer not found! Please train and save them first.")
        return None, None

    clf = joblib.load(model_path,mmap_mode="r")
    vectorizer = joblib.load(vectorizer_path)
    return clf, vectorizer

# 🔹 Load Model & Vectorizer
clf, vectorizer_tfidf = load_model()


# 🔹 Hugging Face API for Treatment Recommendations
HF_API_KEY = "KEY"  # Replace with your actual API key

def get_treatment_recommendation(disease):
    """Query Hugging Face API for AI-powered treatment recommendations."""
    API_URL = "https://api-inference.huggingface.co/models/google/flan-t5-large"
    headers = {"Authorization": f"Bearer {HF_API_KEY}"}
    payload = {"inputs": f"What is the recommended treatment for {disease}?"}

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        if result and isinstance(result, list) and "generated_text" in result[0]:
            return result[0]["generated_text"]
        else:
            return "⚠️ No generated text in response."
    else:
        return f"⚠️ Error: {response.status_code} - {response.text}"

# 🔹 Streamlit UI
st.title("🩺 Disease Prediction & Treatment Recommendations")
st.write("Select your symptoms below to get a predicted disease and treatment suggestion.")

# 🔹 Load Symptoms Dataset
@st.cache_data
def load_symptom_data():
    data_path = os.path.join(drive_path, "processed_diseases-priority.csv")  # Ensure correct file path
    df = pd.read_csv(data_path)
    return df

df = load_symptom_data()

# 🔹 Extract Unique Symptoms
all_symptoms = set()
for symptoms in df["Symptoms"].astype(str).fillna(""):
    all_symptoms.update(symptoms.split(", "))

# 🔹 User Input: Multi-select Symptoms
selected_symptoms = st.multiselect("🩺 Select Symptoms:", sorted(all_symptoms))

if st.button("🔍 Predict Disease & Get Treatment"):
    if selected_symptoms:
        # 🔹 Format symptoms into a single input string
        symptoms_input = ", ".join(selected_symptoms)


        # 🔹 Predict disease using TF-IDF + Random Forest
        input_vector = vectorizer_tfidf.transform([symptoms_input])
        predicted_disease = clf.predict(input_vector)[0]

        # 🔹 Get AI-generated treatment recommendation
        treatment_recommendation = get_treatment_recommendation(predicted_disease)

        # 🔹 Display results
        st.success(f"🩺 **Predicted Disease:** {predicted_disease}")
        st.info(f"💊 **AI-Recommended Treatment:** {treatment_recommendation}")
    else:
        st.warning("⚠️ Please select at least one symptom!")






Overwriting app.py


In [6]:
!npm install localtunnel
!streamlit run app.py --server.address=localhost &>/content/logs.txt &
!npx localtunnel --port 8501 & curl https://loca.lt/mytunnelpassword

⠙⠹⠸⠼⠴⠦
up to date, audited 23 packages in 1s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠦34.53.27.215⠙your url is: https://rich-beers-notice.loca.lt
